In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kendalltau, rankdata, entropy
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
import joblib
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/metadata.json
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/study_final.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_models_info.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_oof_preds.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_predictions.pkl
/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_l2_ridge_0.9557164076282394.csv
/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_base_xgb.csv
/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_base_allcats_xgb.csv
/kaggle/input/notebooks/masa

In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
y = train['Heart Disease']
y_mapped = y.map({'Presence': 1, 'Absence': 0})
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [4]:
oofs_ext_xgb = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_allcats_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb1_ext'})
test_ext_xgb = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_allcats_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb1_ext'})

oofs_ext_realmlp = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_base_allcats_realmlp.csv').drop(columns='id').rename(columns={'Heart Disease': 'realmlp_ext'})
test_ext_realmlp = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_base_allcats_realmlp.csv').drop(columns='id').rename(columns={'Heart Disease': 'realmlp_ext'})

oofs_ext_xgb2 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_base_allcats_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb2_ext'})
test_ext_xgb2 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_base_allcats_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb2_ext'})

oofs_ext_xgb3 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_base_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb3_ext'})
test_ext_xgb3 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_base_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb3_ext'})

oofs_ext_l2 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_l2_ridge_0.9557164076282394.csv').drop(columns='id').rename(columns={'Heart Disease': 'l2_ext'})
test_ext_l2 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_l2_ridge_0.9557164076282394.csv').drop(columns='id').rename(columns={'Heart Disease': 'l2_ext'})

oofs_ext_xgb4 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/oof_te_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb4_ext'})
test_ext_xgb4 = pd.read_csv('/kaggle/input/notebooks/masayakawamata/trust-your-cv-a-robust-ensemble-strategy/test_te_xgb.csv').drop(columns='id').rename(columns={'Heart Disease': 'xgb4_ext'})

In [5]:
oofs_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_oof_preds.npy')
test_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_test_preds.npy')

oofs_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy')
test_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy')

oofs_extreme_preset = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/AUTOGLUON_EXTREME_QUALITY/autogluon_all_oofs (1).csv')
test_extreme_preset = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/AUTOGLUON_EXTREME_QUALITY/autogluon_all_test (1).csv')

oofs_best_preset = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/AUTOGLUON_BEST_QUALITY/autogluon_all_oofs.csv')
test_best_preset = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/AUTOGLUON_BEST_QUALITY/autogluon_all_test.csv')

print(f"gblinear shapes: OOF={oofs_gblinear.shape}, Test={test_gblinear.shape}")
print(f"gbtree shapes: OOF={oofs_gbtree.shape}, Test={test_gbtree.shape}")


oofs_gbtree_df = pd.DataFrame(oofs_gbtree.T)
test_gbtree_df = pd.DataFrame(test_gbtree.T)

oofs_gblinear_df = pd.DataFrame(oofs_gblinear.T)  
test_gblinear_df = pd.DataFrame(test_gblinear.T)

oofs_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(oofs_gblinear_df.shape[1])]
test_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(test_gblinear_df.shape[1])]

oofs_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(oofs_gbtree_df.shape[1])]
test_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(test_gbtree_df.shape[1])]

realmlp_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/oof.csv').drop(columns='id')
realmlp_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/submission.csv').drop(columns='id')

realmlp2_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_realmlp2.csv').drop(columns='id')
realmlp2_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_realmlp2.csv').drop(columns='id')

danet_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_DANet_0.9550176575907665.csv').drop(columns='id')
danet_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_DANet_0.9550176575907665.csv').drop(columns='id')

autoint_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_autoint_0.9553361504101969.csv').drop(columns='id')
autoint_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_autoint_0.9553361504101969.csv').drop(columns='id')

node_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_node_0.9556000217647136.csv').drop(columns='id')
node_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_node_0.9556000217647136.csv').drop(columns='id')

gandalf_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_gandalf_0.9525418820633449.csv').drop(columns='id')
gandalf_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_gandalf_0.9525418820633449.csv').drop(columns='id')

tabnet_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/oof_tabnet_0.9493744672671649.csv').drop(columns='id')
tabnet_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/PYTORCHTABULAR_PYTABKIT/submission_tabnet_0.9493744672671649.csv').drop(columns='id')

resnet50_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/oof_resnet50.csv').drop(columns='id')
resnet50_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/submission_resnet50.csv').drop(columns='id')

tabm_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/oof_TabM_D.csv').drop(columns='id')
tabm_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/submission_TabM_D.csv').drop(columns='id')

cat_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/oof_original_0.955705176736493.csv').drop(columns='Unnamed: 0').rename(columns={'Heart Disease': 'cat_target'})
cat_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/submission_original_0.955705176736493.csv').drop(columns='id').rename(columns={'Heart Disease': 'cat_target'})

fft_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/ftt_transformer/oof_fft_transformer.csv').drop(columns='id').rename(columns={'Heart Disease_prob': 'fft_preds'})
fft_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/ftt_transformer/submission_fft_transformer.csv').drop(columns='id').rename(columns={'Heart Disease': 'fft_preds'})

gblinear shapes: OOF=(24, 630000), Test=(24, 270000)
gbtree shapes: OOF=(120, 630000), Test=(120, 270000)


In [6]:
oofs_df = pd.concat([realmlp_oof.rename(columns={'Heart Disease_prob': 'realmlp_preds'}), 
                     resnet50_oof.rename(columns={'Heart Disease_prob': 'resnet50_preds'}), 
                     tabm_oof.rename(columns={'Heart Disease_prob': 'tabm_preds'}), 
                     realmlp2_oof.rename(columns={'Heart Disease_prob': 'realmlp2_preds'}),
                     danet_oof.rename(columns={'Heart Disease_prob': 'danet_preds'}),
                     autoint_oof.rename(columns={'Heart Disease_prob': 'autoint_preds'}),
                     node_oof.rename(columns={'Heart Disease_prob': 'node_preds'}),
                     gandalf_oof.rename(columns={'Heart Disease_prob': 'gandalf_preds'}),
                     tabnet_oof.rename(columns={'Heart Disease_prob': 'tabnet_preds'}),
                     cat_oof, fft_oof, oofs_extreme_preset, oofs_best_preset,
                     oofs_gbtree_df, oofs_gblinear_df,
                     # oofs_ext_xgb, oofs_ext_xgb2, oofs_ext_xgb3, oofs_ext_xgb4, oofs_ext_realmlp
                    ], axis=1)
test_df = pd.concat([realmlp_test.rename(columns={'Heart Disease': 'realmlp_preds'}), 
                     resnet50_test.rename(columns={'Heart Disease': 'resnet50_preds'}), 
                     tabm_test.rename(columns={'Heart Disease': 'tabm_preds'}), 
                     realmlp2_test.rename(columns={'Heart Disease': 'realmlp2_preds'}),
                     danet_test.rename(columns={'Heart Disease': 'danet_preds'}),
                     autoint_test.rename(columns={'Heart Disease': 'autoint_preds'}),
                     node_test.rename(columns={'Heart Disease': 'node_preds'}),
                     gandalf_test.rename(columns={'Heart Disease': 'gandalf_preds'}),
                     tabnet_test.rename(columns={'Heart Disease': 'tabnet_preds'}),
                     cat_test, fft_test, test_extreme_preset, test_best_preset,
                     test_gbtree_df, test_gblinear_df,
                     # test_ext_xgb, test_ext_xgb2, test_ext_xgb3, test_ext_xgb4, test_ext_realmlp
                    ], axis=1)

# oofs_df['gblinear_oof'] = oofs_gblinear_df['gblinear_model_23'].values
# test_df['gblinear_test'] = test_gblinear_df['gblinear_model_23'].values

In [7]:
print(f"Combined OOFs shape: {oofs_df.shape}")
print(f"Combined Tests shape: {test_df.shape}")

# print(f"\nModel types distribution:")
# print(f"GBLinear models: {len([c for c in oofs_df.columns if 'gblinear' in c])}")
# print(f"GBTree models: {len([c for c in oofs_df.columns if 'gbtree' in c])}")
# print(f"Total models: {all_oofs_df.shape[1]}")

Combined OOFs shape: (630000, 197)
Combined Tests shape: (270000, 197)


In [8]:
from numba import njit, prange

# 1. Numba-Accelerated AUC (The Speed Engine)
@njit
def fast_auc_numba(y_true, y_prob):
    """
    JIT-compiled AUC calculation. 
    This runs at C++ speeds, bypassing Python's slow loops.
    """
    # Sort indices by probability
    order = np.argsort(y_prob)[::-1]
    y_true_sorted = y_true[order]
    
    n_true = np.sum(y_true_sorted)
    n_false = len(y_true_sorted) - n_true
    
    if n_true == 0 or n_false == 0:
        return 0.5
        
    tp = 0
    auc = 0
    # Single pass through the sorted array to calculate area
    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            tp += 1
        else:
            auc += tp
            
    return auc / (n_true * n_false)
    
def select_diverse_models(df_oofs, y_true, n_to_select=15, anchor_col=None, min_auc=0.95):
    """
    Selects models that are both high-performing and diverse.
    Ensures no model with AUC < min_auc is included.
    """
    # 1. Preparation for Numba
    y_raw = y_true.values if hasattr(y_true, 'values') else y_true
    y_raw = y_raw.astype(np.int32).ravel() 
    
    print(f"[*] Filtering models with AUC >= {min_auc}...")
    
    # 2. Calculate individual model performance
    scores = {}
    qualified_cols = []
    for col in df_oofs.columns:
        col_values = df_oofs[col].values.astype(np.float32).ravel()
        score = fast_auc_numba(y_raw, col_values)
        scores[col] = score
        if score >= min_auc:
            qualified_cols.append(col)
            
    print(f"[*] Found {len(qualified_cols)} models meeting the {min_auc} threshold.")
    
    if len(qualified_cols) == 0:
        print("[!] No models met the threshold! Lower the min_auc.")
        return []

    # 3. Determine Anchor Model
    if anchor_col in qualified_cols:
        elite_models = [anchor_col]
        print(f"[*] Starting with Anchor Model: {anchor_col} (AUC: {scores[anchor_col]:.6f})")
    else:
        # Fallback to the best model in the qualified list
        sorted_qualified = sorted([(c, scores[c]) for c in qualified_cols], key=lambda x: x[1], reverse=True)
        elite_models = [sorted_qualified[0][0]]
        print(f"[*] Starting with Best Qualified Model: {elite_models[0]} (AUC: {scores[elite_models[0]]:.6f})")

    remaining_models = [m for m in qualified_cols if m not in elite_models]
    
    # 4. Correlation Analysis (Sampled for speed)
    print("Computing Spearman Correlation Matrix (sampling 50k rows for efficiency)...")
    corr_matrix = df_oofs[qualified_cols].sample(min(50000, len(df_oofs))).corr(method='spearman')
    
    # 5. Greedy Diversity Selection
    # We cap n_to_select by the number of qualified models available
    target_count = min(n_to_select, len(qualified_cols))
    
    while len(elite_models) < target_count:
        best_candidate = None
        lowest_avg_corr = float('inf')
        
        for candidate in remaining_models:
            # Get mean correlation to the existing elite group
            avg_corr = corr_matrix.loc[candidate, elite_models].values.mean()
            
            if avg_corr < lowest_avg_corr:
                lowest_avg_corr = avg_corr
                best_candidate = candidate
        
        if best_candidate:
            elite_models.append(best_candidate)
            remaining_models.remove(best_candidate)
            print(f"Added {len(elite_models)}/{target_count}: {best_candidate} "
                  f"(Avg Corr: {lowest_avg_corr:.4f} | AUC: {scores[best_candidate]:.6f})")
        else:
            break
            
    return elite_models

# --- EXECUTION ---
# Change 'realmlp_preds' to match your actual column name exactly
anchor = 'realmlp_preds' 

elite_columns = select_diverse_models(
    oofs_df, 
    y_mapped, 
    n_to_select=100, 
    anchor_col=anchor, 
    min_auc=0.954
)

oofs_elite = oofs_df[elite_columns]
test_elite = test_df[elite_columns]

print(f"\n[+] Selected {len(elite_columns)} elite models for ensembling.")

[*] Filtering models with AUC >= 0.954...
[*] Found 153 models meeting the 0.954 threshold.
[*] Starting with Anchor Model: realmlp_preds (AUC: 0.955669)
Computing Spearman Correlation Matrix (sampling 50k rows for efficiency)...
Added 2/100: gblinear_model_12 (Avg Corr: 0.9925 | AUC: 0.954135)
Added 3/100: NeuralNetTorch_r79_BAG_L1 (Avg Corr: 0.9925 | AUC: 0.955094)
Added 4/100: gbtree_model_8 (Avg Corr: 0.9924 | AUC: 0.954627)
Added 5/100: NeuralNetTorch_r22_BAG_L1 (Avg Corr: 0.9925 | AUC: 0.954609)
Added 6/100: gblinear_model_3 (Avg Corr: 0.9927 | AUC: 0.954206)
Added 7/100: NeuralNetTorch_BAG_L1 (Avg Corr: 0.9926 | AUC: 0.954938)
Added 8/100: LightGBMLarge_BAG_L1 (Avg Corr: 0.9929 | AUC: 0.954752)
Added 9/100: danet_preds (Avg Corr: 0.9931 | AUC: 0.955018)
Added 10/100: gblinear_model_22 (Avg Corr: 0.9931 | AUC: 0.954263)
Added 11/100: NeuralNetFastAI_r102_BAG_L1 (Avg Corr: 0.9930 | AUC: 0.955114)
Added 12/100: NeuralNetFastAI_BAG_L1 (Avg Corr: 0.9933 | AUC: 0.955116)
Added 13/100:

In [9]:
from tqdm.auto import tqdm
import time


# 2. Optimized Hill Climber
def optimized_hill_climb_final(df_oofs, y_true, batch_size=30, 
                               patience=30, std_dev=0.01, max_steps=1000):
    
    print("""
        \033[91m
        █▄░█ █░█ █▀▄▀█ █▄▄ ▄▀█   █▀▀ █▄░█ █▀▀ █ █▄░█ █▀▀
        █░▀█ █▄█ █░▀░█ █▄█ █▀█   ██▄ █░▀█ █▄█ █ █░▀█ ██▄
        \033[0m
        \033[96m>> Numba JIT Enabled | 150 Models | 600k Rows | No Subsampling\033[0m
        """)

    # Ensure float32 for faster matrix multiplication
    X = df_oofs.values.astype(np.float32)
    X = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X)
    # Robust target conversion
    if hasattr(y_true, 'map'):
        y = y_true.values.astype(np.int32)
    else:
        y = y_true.astype(np.int32)
    
    n_models = X.shape[1]
    
    # Initialize weights
    best_weights = np.ones(n_models) / n_models
    # First prediction
    current_preds = X @ best_weights
    best_score = fast_auc_numba(y, current_preds)
    
    print(f"[*] Initial Baseline ROC-AUC: {best_score:.6f}")
    
    bad_steps = 0
    start_time = time.time()
    pbar = tqdm(range(max_steps), desc="Optimizing Weights")
    
    for step in pbar:
        # print(f'current step: {step}')
        # Generate BATCH of weights at once
        # Nudge only a subset of models (20%) to find a better direction
        deltas = np.random.normal(0, std_dev, size=(batch_size, n_models))
        mask = np.random.rand(batch_size, n_models) > 0.2
        deltas[mask] = 0
        
        trial_weights = np.maximum(0, best_weights + deltas)
        trial_weights /= (trial_weights.sum(axis=1)[:, None] + 1e-12)
        
        # Matrix-Matrix Multiplication (Batch Prediction)
        # This calculates predictions for all 'batch_size' trials in one go
        all_preds = X @ trial_weights.T 
        
        found_better = False
        step_best_score = best_score
        step_best_weights = best_weights
        
        # Use the Numba function to check each trial in the batch
        for i in range(batch_size):
            score = fast_auc_numba(y, all_preds[:, i])
            if score > step_best_score:
                step_best_score = score
                step_best_weights = trial_weights[i].copy()
                found_better = True
        
        if found_better:
            improvement = step_best_score - best_score
            best_score = step_best_score
            best_weights = step_best_weights
            bad_steps = 0
        else:
            improvement = 0
            bad_steps += 1
            
        pbar.set_postfix({
            "AUC": f"{best_score:.6f}", 
            "Improv": f"{improvement:.2e}", 
            "P": f"{bad_steps}/{patience}"
        })
        
        if bad_steps >= patience:
            print(f"\n[!] Early Stopping triggered at step {step}")
            break
            
    print(f"\n[+] Optimization Finished in {time.time() - start_time:.2f}s")
    print(f"[+] Final AUC: {best_score:.6f}")
    
    return best_weights

def hc_with_anchor(df_oofs, y_true, anchor_idx=0, min_anchor_weight=0.80, 
                   batch_size=10, patience=30, std_dev=0.01, max_steps=1000):
    X = df_oofs.values.astype(np.float32)
    y = y_true.values.astype(np.int32) if hasattr(y_true, 'values') else y_true.astype(np.int32)
    n_models = X.shape[1]
    
    # Start with all weight on Anchor
    best_weights = np.zeros(n_models)
    best_weights[anchor_idx] = 1.0
    best_score = fast_auc_numba(y, X @ best_weights)
    
    pbar = tqdm(range(max_steps), desc="Anchor-Protected HC")
    bad_steps = 0
    
    for step in pbar:
        deltas = np.random.normal(0, std_dev, size=(batch_size, n_models))
        # Keep anchor weight stable-ish
        trial_weights = np.maximum(0, best_weights + deltas)
        
        # FORCE the anchor to have at least min_anchor_weight
        # We set it first, then normalize the remaining weights around it
        trial_weights[:, anchor_idx] = np.maximum(min_anchor_weight, trial_weights[:, anchor_idx])
        trial_weights /= trial_weights.sum(axis=1)[:, None]
        
        all_preds = X @ trial_weights.T
        found_better = False
        for i in range(batch_size):
            score = fast_auc_numba(y, all_preds[:, i])
            if score > best_score:
                best_score, best_weights, found_better = score, trial_weights[i], True
                bad_steps = 0
                break
        
        if not found_better: bad_steps += 1
        pbar.set_postfix({"AUC": f"{best_score:.6f}", "P": f"{bad_steps}/{patience}"})
        if bad_steps >= patience: break
            
    return best_weights

def hc_on_ranks(df_oofs, y_true, batch_size=10, patience=30, std_dev=0.01, max_steps=1000):
    # Convert every model's OOFs into Ranks [0, 1]
    print("[*] Converting OOFs to Ranks...")
    X_raw = df_oofs.values.astype(np.float32)
    X_ranked = np.zeros_like(X_raw)
    for i in range(X_raw.shape[1]):
        X_ranked[:, i] = rankdata(X_raw[:, i]) / (X_raw.shape[0] + 1)
    
    y = y_true.values.astype(np.int32) if hasattr(y_true, 'values') else y_true.astype(np.int32)
    n_models = X_ranked.shape[1]
    best_weights = np.ones(n_models) / n_models
    best_score = fast_auc_numba(y, X_ranked @ best_weights)
    
    pbar = tqdm(range(max_steps), desc="Rank-Based HC")
    bad_steps = 0
    for step in pbar:
        deltas = np.random.normal(0, std_dev, size=(batch_size, n_models))
        trial_weights = np.maximum(0, best_weights + deltas)
        trial_weights /= trial_weights.sum(axis=1)[:, None]
        
        all_preds = X_ranked @ trial_weights.T
        found_better = False
        for i in range(batch_size):
            score = fast_auc_numba(y, all_preds[:, i])
            if score > best_score:
                best_score, best_weights, found_better = score, trial_weights[i], True
                bad_steps = 0
                break
        if not found_better: bad_steps += 1
        pbar.set_postfix({"AUC": f"{best_score:.6f}", "P": f"{bad_steps}/{patience}"})
        if bad_steps >= patience: break
            
    return best_weights

from sklearn.model_selection import KFold

def bagged_hill_climb(df_oofs, y_true, n_folds=5, max_steps=500):
    X = df_oofs.values.astype(np.float32)
    y = y_true.values.astype(np.int32) if hasattr(y_true, 'values') else y_true.astype(np.int32)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_weights = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f"\n[*] Optimizing Fold {fold+1}/{n_folds}...")
        X_train, y_train = X[train_idx], y[train_idx]
        
        # Use a simplified version of your HC here
        best_w = optimized_hill_climb_final(pd.DataFrame(X_train), y_train, max_steps=max_steps)
        fold_weights.append(best_w)
        
    avg_weights = np.mean(fold_weights, axis=0)
    avg_weights /= avg_weights.sum()
    
    print("\n[+] Bagging Complete. Averaged weights across folds.")
    return avg_weights

# --- RUNNING THE CELL ---
# Ensure y_mapped is 0s and 1s
# best_weights = optimized_hill_climb_final(oofs_df, y_mapped)

In [10]:
%%time
best_weights = optimized_hill_climb_final(
    df_oofs=oofs_elite, 
    y_true=y_mapped, 
     batch_size=1, 
    patience=1000, std_dev=0.02, max_steps=30000
)


        
        █▄░█ █░█ █▀▄▀█ █▄▄ ▄▀█   █▀▀ █▄░█ █▀▀ █ █▄░█ █▀▀
        █░▀█ █▄█ █░▀░█ █▄█ █▀█   ██▄ █░▀█ █▄█ █ █░▀█ ██▄
        
        >> Numba JIT Enabled | 150 Models | 600k Rows | No Subsampling
        
[*] Initial Baseline ROC-AUC: 0.955716


Optimizing Weights:   0%|          | 0/30000 [00:00<?, ?it/s]


[!] Early Stopping triggered at step 5769

[+] Optimization Finished in 1010.39s
[+] Final AUC: 0.955781
CPU times: user 57min 19s, sys: 4.58 s, total: 57min 24s
Wall time: 16min 59s


In [11]:
oofs_elite['hc_feature'] = oofs_elite @ best_weights
test_elite['hc_feature'] = test_elite @ best_weights

In [12]:
# %%time
# anchor_idx = list(oofs_elite.columns).index('realmlp_preds')

# weights_anchor = hc_with_anchor(
#     oofs_elite, 
#     y_mapped, 
#     anchor_idx=anchor_idx, 
#     min_anchor_weight=0.75, 
#     batch_size=5, 
#     patience=100, 
#     std_dev=0.01, 
#     max_steps=2
# )

In [13]:
# %%time
# weights_rank = hc_on_ranks(
#     oofs_elite, 
#     y_mapped, 
#     batch_size=5, 
#     patience=100, 
#     std_dev=0.01, 
#     max_steps=2000
# )

In [14]:
# %%time

# weights_bagged = bagged_hill_climb(
#     oofs_elite, 
#     y_mapped, 
#     n_folds=5,
#     max_steps=600
# )

In [15]:
# best_weights

In [16]:
# final_preds = test_elite.values @ weights_bagged

In [17]:
# # Geometric Blend (Log-space)
# # final_preds = np.exp(0.65 * np.log(realmlp_test) + 0.35 * np.log(tabm_test))
# final_preds = (rankdata(realmlp_test) * 0.72 + rankdata(fft_test) * 0.28) / 2 

In [18]:
# # test_preds = test_gbtree_df['gbtree_model_59'].values
# sample_submission[CONFIG.TARGET] = final_preds
# sample_submission.to_csv('submission_csv_hillclimbing.csv', index=False)

In [19]:
# sample_submission

In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm
import numpy as np

# Your data
X_meta_train = oofs_elite.values
X_meta_test = test_elite.values
y = train[CONFIG.TARGET].map(class_mapping).values

strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
le = LabelEncoder()
stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))

print(f"Meta-train shape: {X_meta_train.shape}")
print(f"Meta-test shape: {X_meta_test.shape}")

# Stratified K-Fold setup
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, shuffle=True, random_state=CONFIG.SEED)

# Storage for predictions
meta_oof_preds = np.zeros(len(X_meta_train))
meta_test_preds = np.zeros(len(X_meta_test))
fold_scores = []

print(f"\nTraining meta-model with {CONFIG.N_FOLDS}-fold CV")
print("="*60)

# CV loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, y), 1):
    
    print(f"\nFold {fold}:")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    # Split meta-data
    X_train_fold = X_meta_train[train_idx]
    X_val_fold = X_meta_train[val_idx]
    y_train_fold = y[train_idx]
    y_val_fold = y[val_idx]
    X_train_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_train_fold)
    X_val_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_val_fold)
    X_test_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_meta_test)

    # Normalize ranks to [0,1] (important before logit)
    # n_train = X_train_fold.shape[0]
    # n_val   = X_val_fold.shape[0]
    # n_test  = X_meta_test.shape[0]
    # X_train_ranked = (X_train_ranked - 1) / (n_train - 1)
    # X_val_ranked   = (X_val_ranked - 1) / (n_val - 1)
    # X_test_ranked  = (X_test_ranked - 1) / (n_test - 1)
    
    # # 2. LOGIT TRANSFORM (safe)
    # eps = 1e-15
    # X_train_logit = np.log(np.clip(X_train_ranked, eps, 1-eps) / (1 - np.clip(X_train_ranked, eps, 1-eps)))
    # X_val_logit   = np.log(np.clip(X_val_ranked, eps, 1-eps) / (1 - np.clip(X_val_ranked, eps, 1-eps)))
    # X_test_logit  = np.log(np.clip(X_test_ranked, eps, 1-eps) / (1 - np.clip(X_test_ranked, eps, 1-eps)))
    # ===== FIXED: USE RIDGE, NOT LOGISTIC (for now) =====
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_ranked)
    X_val_scaled = scaler.transform(X_val_ranked)
    X_test_scaled = scaler.transform(X_test_ranked)
    
    # ===== OPTION A: RIDGE REGRESSION (WORKS) =====
    # ridge = Ridge(alpha=6.4454853482060432, random_state=CONFIG.SEED + fold)
    ridge = LogisticRegression(
            C=0.2,                    # Moderate regularization (try 0.01-1.0)
            penalty='l2',             # L2 works well for correlated features
            solver='liblinear',        # Fast and works with L2
            class_weight='balanced',   # Since your target might be imbalanced
            max_iter=2000,             # Enough iterations for convergence
            random_state=CONFIG.SEED + fold,
            fit_intercept=True         # Important with scaled inputs
        )
    ridge.fit(X_train_scaled, y_train_fold)
    
    # Predict
    val_preds = ridge.predict_proba(X_val_scaled)[:,1]
    test_preds = ridge.predict_proba(X_test_scaled)[:,1]

    # val_preds = ridge.predict(X_val_scaled)
    # test_preds = ridge.predict(X_test_scaled)
    
    # ===== OPTION B: FIXED LOGISTIC REGRESSION =====
    # Uncomment this if you want to try logistic
    # logreg = LogisticRegression(
    #     C=0.01,  # STRONG REGULARIZATION (1/alpha)
    #     penalty='l2',
    #     solver='lbfgs',  # BETTER FOR LARGE DATASETS
    #     max_iter=1000,
    #     random_state=CONFIG.SEED + fold
    # )
    # logreg.fit(X_train_scaled, y_train_fold)
    # val_preds = logreg.predict_proba(X_val_scaled)[:, 1]
    # test_preds = logreg.predict_proba(X_test_scaled)[:, 1]
    
    # Clip predictions to [0, 1]
    # val_preds = np.clip(val_preds, 0, 1)
    # test_preds = np.clip(test_preds, 0, 1)
    
    # Check for weird predictions
    print(f"  Val preds range: [{val_preds.min():.4f}, {val_preds.max():.4f}]")
    print(f"  Mean val pred: {val_preds.mean():.4f}")
    
    # Store predictions
    meta_oof_preds[val_idx] = val_preds
    meta_test_preds += test_preds / CONFIG.N_FOLDS
    
    # Score
    fold_score = roc_auc_score(y_val_fold, val_preds)
    fold_scores.append(fold_score)
    
    print(f"  Fold {fold} AUC: {fold_score:.6f}")
    print(f"  Ridge coef stats: mean={ridge.coef_.mean():.6f}, std={ridge.coef_.std():.6f}")

# ===== FINAL RESULTS =====
print(f"\n{'='*60}")
print("META-MODEL CV RESULTS")
print(f"{'='*60}")

print(f"Fold scores: {[f'{s:.6f}' for s in fold_scores]}")
print(f"Mean fold score: {np.mean(fold_scores):.6f} (±{np.std(fold_scores):.6f})")

# Check for weird predictions in final OOF
print(f"\nFinal OOF predictions analysis:")
print(f"  Range: [{meta_oof_preds.min():.6f}, {meta_oof_preds.max():.6f}]")
print(f"  Mean: {meta_oof_preds.mean():.6f}")
print(f"  Std: {meta_oof_preds.std():.6f}")

# Final score
final_score = roc_auc_score(y, meta_oof_preds)
print(f"\nOOF AUC Score: {final_score:.6f}")

# Compare with simple average
simple_avg_preds = np.mean(np.apply_along_axis(rankdata, 0, X_meta_train), axis=1)
simple_avg_score = roc_auc_score(y, simple_avg_preds)
print(f"Simple average baseline: {simple_avg_score:.6f}")
print(f"Meta-model improvement: +{final_score - simple_avg_score:.6f}")

# Create submission
sample_submission['Heart Disease'] = meta_test_preds
sample_submission.to_csv(f'submission_meta_lr_ranked_{final_score:.6f}.csv', index=False)
print(f"\n✅ Saved: submission_meta_lr_ranked_{final_score:.6f}.csv")

Meta-train shape: (630000, 101)
Meta-test shape: (270000, 101)

Training meta-model with 5-fold CV

Fold 1:
Train size: 504000, Val size: 126000
  Val preds range: [0.0016, 0.0315]
  Mean val pred: 0.0101
  Fold 1 AUC: 0.956164
  Ridge coef stats: mean=0.034392, std=0.084477

Fold 2:
Train size: 504000, Val size: 126000
  Val preds range: [0.0016, 0.0311]
  Mean val pred: 0.0099
  Fold 2 AUC: 0.955023
  Ridge coef stats: mean=0.034514, std=0.097891

Fold 3:
Train size: 504000, Val size: 126000
  Val preds range: [0.0016, 0.0314]
  Mean val pred: 0.0101
  Fold 3 AUC: 0.955870
  Ridge coef stats: mean=0.034429, std=0.097349

Fold 4:
Train size: 504000, Val size: 126000
  Val preds range: [0.0016, 0.0313]
  Mean val pred: 0.0100
  Fold 4 AUC: 0.955513
  Ridge coef stats: mean=0.034462, std=0.098036

Fold 5:
Train size: 504000, Val size: 126000
  Val preds range: [0.0016, 0.0315]
  Mean val pred: 0.0101
  Fold 5 AUC: 0.956313
  Ridge coef stats: mean=0.034378, std=0.093469

META-MODEL CV R

In [21]:
# meta_test_preds = np.zeros(len(X_meta_test))
# X_train_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_meta_train)
#     # X_val_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_val_fold)
# X_test_ranked = np.apply_along_axis(lambda x: rankdata(x, method='average'), 0, X_meta_test)

#     # Normalize ranks to [0,1] (important before logit)
#     # n_train = X_train_fold.shape[0]
#     # n_val   = X_val_fold.shape[0]
#     # n_test  = X_meta_test.shape[0]
#     # X_train_ranked = (X_train_ranked - 1) / (n_train - 1)
#     # X_val_ranked   = (X_val_ranked - 1) / (n_val - 1)
#     # X_test_ranked  = (X_test_ranked - 1) / (n_test - 1)
    
#     # # 2. LOGIT TRANSFORM (safe)
#     # eps = 1e-15
#     # X_train_logit = np.log(np.clip(X_train_ranked, eps, 1-eps) / (1 - np.clip(X_train_ranked, eps, 1-eps)))
#     # X_val_logit   = np.log(np.clip(X_val_ranked, eps, 1-eps) / (1 - np.clip(X_val_ranked, eps, 1-eps)))
#     # X_test_logit  = np.log(np.clip(X_test_ranked, eps, 1-eps) / (1 - np.clip(X_test_ranked, eps, 1-eps)))
#     # ===== FIXED: USE RIDGE, NOT LOGISTIC (for now) =====
#     # Scale features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train_ranked)
# # X_val_scaled = scaler.transform(X_val_ranked)
# X_test_scaled = scaler.transform(X_test_ranked)
    
#     # ===== OPTION A: RIDGE REGRESSION (WORKS) =====
#     # ridge = Ridge(alpha=6.4454853482060432, random_state=CONFIG.SEED + fold)
# ridge = LogisticRegression(
#             C=0.2,                    # Moderate regularization (try 0.01-1.0)
#             penalty='l2',             # L2 works well for correlated features
#             solver='liblinear',        # Fast and works with L2
#             # class_weight='balanced',   # Since your target might be imbalanced
#             max_iter=2000,             # Enough iterations for convergence
#             random_state=CONFIG.SEED,
#             fit_intercept=True         # Important with scaled inputs
#         )
# ridge.fit(X_train_scaled, y)
# test_preds = ridge.predict_proba(X_test_scaled)[:,1]

# meta_test_preds += test_preds / CONFIG.N_FOLDS

# sample_submission['Heart Disease'] = meta_test_preds
# sample_submission.to_csv(f'submission_meta_lr_ranked.csv', index=False)